<div dir="rtl" align="right">

# تحميلُ مجموعةِ بياناتٍ من MOABB

**مجموعةُ البياناتِ**: BNCI2014-001 (تَخيّلٌ حركيٌّ)
**المُشاركُ**: 1
**النموذجُ**: MotorImagery (n_classes=2)
**القنواتُ**: 22 EEG
**معدّلُ أخذِ العيناتِ**: 250 Hz

---

## نظرةٌ عامّةٌ

MOABB (Mother of All BCI Benchmarks) تُوفّرُ وصولاً مُوحّداً إلى مجموعاتِ بياناتِ BCI العامة. هذا الدفترُ يحمّلُ مجموعةَ بياناتِ BNCI2014-001 لِتَخيّلٍ حركيٍّ لِلمُشاركِ 1، ويَستخرجُ الحقبَ بِنموذجِ MotorImagery، ويُصوّرُ عددَ المحاولاتِ لِكلِّ فئةٍ.

## ماذا يَفعلُ هذا الدفترُ

- يحمّلُ BNCI2014-001 لِلمُشاركِ 1 عبرَ MOABB (تُنزّلُ البياناتُ تلقائياً عندَ أولِ تشغيلٍ)
- يَستخرجُ الحقبَ بِاستخدامِ نموذجِ MotorImagery
- يَطبعُ بنيةَ مجموعةِ البياناتِ: الجلساتِ، التشغيلاتِ، القنواتِ، معدّلَ أخذِ العيناتِ
- يَعُدُّ المحاولاتِ لِكلِّ فئةٍ ويَرسمُ مخططاً شريطياً

## المُخرجاتُ المُتوقّعةُ

- جلستانِ بِستِّ تشغيلاتٍ لِكلٍّ منهما
- 22 قناةَ EEG بِمعدّلِ 250 Hz
- 576 حقبةً إجمالاً على 4 فئاتٍ (144 لِكلٍّ)
- مخططٌ شريطيٌّ يُظهرُ أعداداً مُتوازنةً من المحاولاتِ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| dataset | BNCI2014_001 | مجموعةُ بياناتِ تَخيّلٍ حركيٍّ من MOABB |
| subjects | [1] | المُشاركُ 1 فقط |
| n_classes | 2 | فئاتُ نموذجِ MotorImagery |
| sfreq | 250 Hz | معدّلُ أخذِ العيناتِ |


</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

MOABB تُنزّلُ البياناتِ تلقائياً عندَ أولِ استخدامٍ (~44 ميجابايت لِلمُشاركِ 1). التشغيلاتُ اللاحقةُ تَستخدمُ البياناتِ المُخزّنةَ.


</div>


In [ ]:
from moabb.datasets import BNCI2014_001
ds = BNCI2014_001()
sessions = ds.get_data(subjects=[1])
subject_key = list(sessions.keys())[0]
session_dict = sessions[subject_key]
n_sessions = len(session_dict)
n_runs = len(next(iter(session_dict.values())))
first_run = next(iter(next(iter(session_dict.values())).values()))
n_channels_raw = len(first_run.ch_names)
sfreq = first_run.info['sfreq']
print(f'Subject 1: {n_sessions} sessions, {n_runs} runs/session')
print(f'Raw channels: {n_channels_raw}, Sampling rate: {sfreq} Hz')
print(f'Channel names: {first_run.ch_names}')



In [ ]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[1])
print(f'X shape: {X.shape}  (n_trials, n_channels, n_samples)')
print(f'Labels shape: {labels.shape}')
print(f'Meta shape: {meta.shape}')



<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

نَطبعُ أشكالَ الحقبِ، والوسومَ، وعددَ المحاولاتِ لِكلِّ فئةٍ.


</div>


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
print(f'Epoch channels: {X.shape[1]}')
print(f'Epoch samples: {X.shape[2]}')
print(f'Epoch duration: {X.shape[2] / sfreq:.2f} s')
print(f'Unique labels: {list(unique_labels)}')
print(f'Trials per class: {dict(zip(unique_labels, counts))}')
print(f'Total trials: {X.shape[0]}')



<div dir="rtl" align="right">

## 4. تطبيقُ التحليلِ

نَعُدُّ المحاولاتِ لِكلِّ فئةٍ لِتجهيزِ المخططِ الشريطيِّ.


</div>


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
trial_counts = dict(zip(unique_labels, counts))
print(f'Trial counts: {trial_counts}')



<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- جميعُ الفئاتِ الأربعِ لها أعدادٌ مُتساويةٌ من المحاولاتِ (مجموعةُ بياناتٍ مُتوازنةٌ)
- المخططُ الشريطيُّ يُظهرُ 144 محاولةً لِكلِّ فئةٍ
- مرّرْ فوقَ الأعمدةِ لِرؤيةِ القيمِ الدقيقةِ



</div>


In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=[go.Bar(x=list(trial_counts.keys()), y=list(trial_counts.values()),
                             marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
                             text=list(trial_counts.values()), textposition='auto')])
fig.update_layout(title='Trials per class - BNCI2014-001 (subject 1)',
                  xaxis_title='Class label', yaxis_title='Number of trials',
                  height=500)
fig.show()



<div dir="rtl" align="right">

## خلاصةٌ

- MOABB تُوفّرُ وصولاً مُوحّداً إلى مجموعاتِ بياناتِ BCI بِواجهةٍ بسيطةٍ
- BNCI2014-001 تَحتوي على جلستينِ، 6 تشغيلاتٍ، 22 قناةَ EEG بِمعدّلِ 250 Hz
- نموذجُ MotorImagery يَستخرجُ حقباً مُتوازنةً (144 لِكلِّ فئةٍ)
- البياناتُ تُنزّلُ تلقائياً وتُخزّنُ لِلاستخدامِ المستقبليِّ



</div>
